# 00 · Introduction to M-Lab Monthly Stats Dataset

This page explains the data model,
terminology, and important caveats that apply across every Monthly Stats Data Slice.

---

## What is Measurement Lab?

[M-Lab](https://www.measurementlab.net/) is an open distributed platform that hosts
the **NDT (Network Diagnostic Tool)** speed test. Millions of tests run daily from
real user devices worldwide. M-Lab publishes all raw results openly and at various aggregation levels, check out our [data catalog](https://measurementlab.net/datasets).

## How does M-Lab use the Monthly Stats Data

These data are used at the input for our **Internet Quality Barometer (IQB)** project.  We aggregate NDT results into monthly,
percentile-based summaries at several geographic granularities. Instead of querying
milliona or billions of raw rows of test daate in BigQuery, you download a small file (~1–5 MB) that
already contains p1–p99 values for each metric, each geography, and each month.

## The Four Metrics

| Metric | Column prefix | Unit | Better direction |
|--------|--------------|------|-----------------|
| Download throughput | `download_p*` | Mbit/s | ↑ Higher |
| Upload throughput | `upload_p*` | Mbit/s | ↑ Higher |
| Latency (min RTT) | `latency_p*` | ms | ↓ Lower |
| Packet loss rate | `loss_p*` | fraction 0–1 | ↓ Lower |

> **Percentile polarity caveat** — For latency and loss the percentile direction is
> *inverted* so that a higher percentile always means "better quality":
> - `latency_p95` = 5th percentile of actual latency (fastest/lowest RTT connections)
> - `latency_p5` = 95th percentile of actual latency (slowest/highest RTT connections)
>
> Keep this in mind when labelling axes or comparing percentile slices across metrics.

## Geographic Slices

| Slice name | Rows represent |
|-----------|---------------|
| `downloads_by_country` | One row per country |
| `downloads_by_country_asn` | Country × ISP |
| `downloads_by_country_subdivision1` | Country × state/province |
| `downloads_by_country_subdivision1_asn` | Country × region × ISP |
| `downloads_by_country_city` | Country × city |
| `downloads_by_country_city_asn` | Country × city × ISP |

Upload slices follow the same structure (`uploads_by_country`, etc.).

> **Date coverage caveat** — Not all slices are updated at the same time.
> Country-level data often leads city/ASN-level data by 1–2 months. Always derive
> your example month as the most recent month present in *all* slices you plan to join.

## On-Disk Cache

These notebooks use a disk cache matching the layout used by the
[IQB library](https://github.com/m-lab/iqb/tree/main/data):

```
./cache/v1/{start_timestamp}/{end_timestamp}/{slice_name}/data.parquet
```

A file is downloaded once and reused in every subsequent session. If you also
have the IQB library installed locally, both tools share the same cache.

## Setup

In [1]:
# ── Dependency check ─────────────────────────────────────────────────────────
# These notebooks do not auto-install packages. If any import below fails,
# install the missing package in your environment first, e.g.:
#   pip install pandas pyarrow requests matplotlib seaborn ipywidgets

import importlib, sys

REQUIRED = {
    "pandas":     "pandas",
    "pyarrow":    "pyarrow",
    "requests":   "requests",
    "matplotlib": "matplotlib",
    "seaborn":    "seaborn",
    "ipywidgets": "ipywidgets",
}

missing = [pip for mod, pip in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("\u26a0\ufe0f  Missing packages — install them before continuing:\n")
    print(f"    pip install {' '.join(missing)}\n")
    raise ImportError(f"Missing: {missing}")
else:
    print("\u2713 All dependencies present.")

✓ All dependencies present.


In [2]:
import json
from io import BytesIO
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import requests
import seaborn as sns
from IPython.display import display, clear_output

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

Matplotlib is building the font cache; this may take a moment.


In [3]:
# ── Load the Monthly Stats manifest ─────────────────────────────────────────────────────
#
# M-Lab publishes a JSON manifest listing every available Monthly Stats parquet file.
# Each entry covers one calendar month at one geographic granularity (slice).
#
# Path format:  cache/v1/{start_timestamp}/{end_timestamp}/{slice_name}/data.parquet
#

MANIFEST_URL = "https://measurementlab.net/data/iqb/manifest.json"
resp = requests.get(MANIFEST_URL, timeout=30)
resp.raise_for_status()
manifest = resp.json()

records = []
for path, meta in manifest["files"].items():
    parts = path.split("/")
    if len(parts) < 6:
        continue
    # 6-segment path: cache / version / start_ts / end_ts / slice / filename
    _, version, start_raw, end_raw, slice_name, filename = parts[:6]
    if filename != "data.parquet":
        continue
    records.append({
        "start":      pd.to_datetime(start_raw, format="%Y%m%dT%H%M%SZ"),
        "end":        pd.to_datetime(end_raw,   format="%Y%m%dT%H%M%SZ"),
        "slice":      slice_name,
        "url":        meta["url"],
        # Canonical local path matching the IQB library's cache layout:
        #   ./cache/v1/{start}/{end}/{slice}/data.parquet
        "cache_path": path,
    })

catalog = (
    pd.DataFrame(records)
    .sort_values(["slice", "start"])
    .reset_index(drop=True)
)

print(f"Catalog loaded — {len(catalog)} entries across {catalog['slice'].nunique()} slices.")
print(f"Full date range: {catalog['start'].min().date()} → {catalog['start'].max().date()}")

Catalog loaded — 2450 entries across 12 slices.
Full date range: 2009-01-01 → 2026-01-01


## Explore the Catalog

Data are available for uploads and downloads independently. Statistics are computed at six levels:

1. country
2. country x asn
3. country x city
4. country x city x asn
5. country x subdivision1,
6. country x subdivision1 x asn, 

Use the dropdown to inspect available monthly data per slice.

In [4]:
slice_summary = (
    catalog.groupby("slice")
    .agg(first_month=("start","min"), last_month=("start","max"),
         months=("start","count"))
    .reset_index()
)

w_slice = widgets.Dropdown(
    options=sorted(catalog["slice"].unique()), description="Slice:",
    layout=widgets.Layout(width="500px"),
)
out = widgets.Output()

def show(change=None):
    with out:
        clear_output(wait=True)
        s = w_slice.value
        r = slice_summary[slice_summary["slice"] == s].iloc[0]
        months = catalog[catalog["slice"] == s]["start"].dt.strftime("%Y-%m-%d").tolist()
        print(f"Slice:            {s}")
        print(f"First month:      {r.first_month.date()}")
        print(f"Latest month:     {r.last_month.date()}")
        print(f"Months available: {r.months}")
        print()
        for i in range(0, len(months), 6):
            print("  " + "  ".join(months[i:i+6]))

w_slice.observe(show, "value")
display(widgets.VBox([w_slice, out]))
show()

---
## Next Steps

- **01-country-level.ipynb** — Compare countries on all four metrics
- **02-asn-isp.ipynb** — Rank ISPs within a country
- **03-subdivisions.ipynb** — State / province breakdown
- **04-subdivision-asn.ipynb** — ISPs by region
- **05-cities.ipynb** — City-level quality
- **06-time-series.ipynb** — Track trends over 12 months